**Stage 1.** Converts YOLOv8x-Pose 17-joint output into an OpenPose-style layout with a synthetic neck and mid-hip.

Known issue: `yolo_to_18_joints()` actually returns **19** joints (neck *and* mid-hip are both added), while `GaitSTGCN` and DenseGait use 18. See the README's Implementation notes.

In [ ]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

# Check if CUDA (GPU support) is available
if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
from IPython.display import Video
import cv2
import numpy as np
import json


#!/usr/bin/env python3
"""
18-Joint Pose Estimator (OpenPose/AlphaPose format)
Extracts poses from video and saves to JSON
"""

import cv2
import numpy as np
import json
# import argparse # Removed argparse
import sys

try:
    from ultralytics import YOLO
except ImportError:
    print("ERROR: Install ultralytics with: pip install ultralytics opencv-python")
    sys.exit(1)

# 18-joint mapping (OpenPose/AlphaPose format)
# YOLO-Pose has 17 joints, we'll add neck as midpoint of shoulders
JOINT_NAMES_18 = [
    'Nose',           # 0
    'Neck',           # 1  (calculated as midpoint of shoulders)
    'RShoulder',      # 2
    'RElbow',         # 3
    'RWrist',         # 4
    'LShoulder',      # 5
    'LElbow',         # 6
    'LWrist',         # 7
    'MidHip',         # 8  (calculated as midpoint of hips)
    'RHip',           # 9
    'RKnee',          # 10
    'RAnkle',         # 11
    'LHip',           # 12
    'LKnee',          # 13
    'LAnkle',         # 14
    'REye',           # 15
    'LEye',           # 16
    'REar',           # 17
    'LEar'            # 18 (note: index 17, total 18 joints from 0-17)
]

# Skeleton connections for 18-joint format
SKELETON_18 = [
    (0, 1),    # Nose -> Neck
    (1, 2),    # Neck -> RShoulder
    (2, 3),    # RShoulder -> RElbow
    (3, 4),    # RElbow -> RWrist
    (1, 5),    # Neck -> LShoulder
    (5, 6),    # LShoulder -> LElbow
    (6, 7),    # LElbow -> LWrist
    (1, 8),    # Neck -> MidHip
    (8, 9),    # MidHip -> RHip
    (9, 10),   # RHip -> RKnee
    (10, 11),  # RKnee -> RAnkle
    (8, 12),   # MidHip -> LHip
    (12, 13),  # LHip -> LKnee
    (13, 14),  # LKnee -> LAnkle
    (0, 15),   # Nose -> REye
    (15, 17),  # REye -> REar
    (0, 16),   # Nose -> LEye
    (16, 18)   # LEye -> LEar (index 17)
]

def yolo_to_18_joints(yolo_keypoints):
    """
    Convert YOLO-Pose 17 keypoints to 18-joint OpenPose/AlphaPose format.

    YOLO format (17 joints):
    0:nose, 1:left_eye, 2:right_eye, 3:left_ear, 4:right_ear,
    5:left_shoulder, 6:right_shoulder, 7:left_elbow, 8:right_elbow,
    9:left_wrist, 10:right_wrist, 11:left_hip, 12:right_hip,
    13:left_knee, 14:right_knee, 15:left_ankle, 16:right_ankle

    Returns 18 joints in OpenPose/AlphaPose format
    """
    joints_18 = []

    # Joint 0: Nose
    joints_18.append(yolo_keypoints[0])

    # Joint 1: Neck (midpoint of shoulders)
    left_shoulder = yolo_keypoints[5]
    right_shoulder = yolo_keypoints[6]
    if left_shoulder['score'] > 0.3 and right_shoulder['score'] > 0.3:
        neck = {
            'x': (left_shoulder['x'] + right_shoulder['x']) / 2,
            'y': (left_shoulder['y'] + right_shoulder['y']) / 2,
            'score': (left_shoulder['score'] + right_shoulder['score']) / 2
        }
    else:
        neck = {'x': 0, 'y': 0, 'score': 0.0}
    joints_18.append(neck)

    # Joint 2: RShoulder (YOLO right_shoulder)
    joints_18.append(yolo_keypoints[6])

    # Joint 3: RElbow (YOLO right_elbow)
    joints_18.append(yolo_keypoints[8])

    # Joint 4: RWrist (YOLO right_wrist)
    joints_18.append(yolo_keypoints[10])

    # Joint 5: LShoulder (YOLO left_shoulder)
    joints_18.append(yolo_keypoints[5])

    # Joint 6: LElbow (YOLO left_elbow)
    joints_18.append(yolo_keypoints[7])

    # Joint 7: LWrist (YOLO left_wrist)
    joints_18.append(yolo_keypoints[9])

    # Joint 8: MidHip (midpoint of hips)
    left_hip = yolo_keypoints[11]
    right_hip = yolo_keypoints[12]
    if left_hip['score'] > 0.3 and right_hip['score'] > 0.3:
        mid_hip = {
            'x': (left_hip['x'] + right_hip['x']) / 2,
            'y': (left_hip['y'] + right_hip['y']) / 2,
            'score': (left_hip['score'] + right_hip['score']) / 2
        }
    else:
        mid_hip = {'x': 0, 'y': 0, 'score': 0.0}
    joints_18.append(mid_hip)

    # Joint 9: RHip (YOLO right_hip)
    joints_18.append(yolo_keypoints[12])

    # Joint 10: RKnee (YOLO right_knee)
    joints_18.append(yolo_keypoints[14])

    # Joint 11: RAnkle (YOLO right_ankle)
    joints_18.append(yolo_keypoints[16])

    # Joint 12: LHip (YOLO left_hip)
    joints_18.append(yolo_keypoints[11])

    # Joint 13: LKnee (YOLO left_knee)
    joints_18.append(yolo_keypoints[13])

    # Joint 14: LAnkle (YOLO left_ankle)
    joints_18.append(yolo_keypoints[15])

    # Joint 15: REye (YOLO right_eye)
    joints_18.append(yolo_keypoints[2])

    # Joint 16: LEye (YOLO left_eye)
    joints_18.append(yolo_keypoints[1])

    # Joint 17: REar (YOLO right_ear)
    joints_18.append(yolo_keypoints[4])

    # Joint 18: LEar (YOLO left_ear) - index 17
    joints_18.append(yolo_keypoints[3])

    return joints_18


def extract_poses(video_path, output_json='pose_18joints.json',
                 model_name='yolov8x-pose.pt'):
    """Extract 18-joint poses from video."""

    print("=" * 60)
    print("18-JOINT POSE EXTRACTION (OpenPose/AlphaPose format)")
    print("=" * 60)
    print(f"\nInput: {video_path}")
    print(f"Output: {output_json}")
    print(f"Model: {model_name}")

    # Load YOLO model
    print("\nLoading YOLO-Pose model...")
    model = YOLO(model_name)
    print("✓ Model loaded")

    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Cannot open {video_path}")
        return False

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"\nVideo: {width}x{height} @ {fps:.2f}fps, {total_frames} frames")
    print("\nProcessing frames...")

    all_frames_data = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_data = []

        # Detect poses
        results = model(frame, verbose=False)

        for r in results:
            if r.keypoints and r.keypoints.data.numel() > 0:
                for person_kpts in r.keypoints.data.cpu().numpy():
                    # Convert YOLO 17 joints to our format
                    yolo_joints = []
                    for kp in person_kpts:
                        x, y, score = kp.tolist()
                        yolo_joints.append({'x': float(x), 'y': float(y), 'score': float(score)})

                    # Convert to 18-joint format
                    joints_18 = yolo_to_18_joints(yolo_joints)

                    frame_data.append({'keypoints': joints_18})

        all_frames_data.append(frame_data)

        frame_idx += 1
        if frame_idx % 30 == 0 or frame_idx == total_frames:
            progress = (frame_idx / total_frames * 100) if total_frames > 0 else 0
            print(f"Progress: {frame_idx}/{total_frames} ({progress:.1f}%) рассчитано", end='\r') # Changed 'Progress: {frame_idx}/{total_frames} ({progress:.1f}%)', end='\r' to include 'рассчитано'

    cap.release()

    # Save JSON
    print(f"\n\nSaving to {output_json}...")
    with open(output_json, 'w') as f:
        json.dump(all_frames_data, f, indent=2)

    # Statistics
    total_detections = sum(len(frame) for frame in all_frames_data)
    avg_persons = total_detections / len(all_frames_data) if all_frames_data else 0

    print(f"✓ Done!")
    print(f"\nStatistics:")
    print(f"  Frames: {len(all_frames_data)}")
    print(f"  Total detections: {total_detections}")
    print(f"  Avg persons/frame: {avg_persons:.2f}")
    print(f"  Format: 18 joints (OpenPose/AlphaPose)")
    print("=" * 60)

    return True


# Removed main() and argparse, direct call for Colab
# if __name__ == "__main__":
#     main()

# You can now define your video path and other parameters here and call extract_poses directly
video_path = 'vid3-matlab.avi' # Replace with your video file path
output_json_file = 'pose_18joints.json'
pose_model = 'yolov8x-pose.pt'

success = extract_poses(video_path, output_json_file, pose_model)

if success:
    print(f"\n✓ Ready for overlay!")
    print(f"\nNext: python pose_overlay_18joints.py \\\n         --inpainted YOUR_VIDEO.mp4 \\\n         --poses {output_json_file}")
else:
    sys.exit(1)
